# 08 — Analytics: 10 Queries de Portfolio (Spark SQL)

10 queries analíticas demonstrando o poder do modelo dimensional com Spark SQL puro.

| # | Query | Técnica |
|---|-------|---------|
| 1 | Receita bruta vs líquida por categoria | GROUP BY + métricas |
| 2 | Top 10 clientes por NetRevenue | ORDER BY + LIMIT |
| 3 | Performance de entrega por Shipper | Accumulating Snapshot |
| 4 | Análise SCD2: receita pelo país histórico | SCD2 awareness |
| 5 | Histórico de preço por produto | Versões SCD2 |
| 6 | Hierarquia de funcionários: receita por gestor | Flattened hierarchy |
| 7 | Sazonalidade: NetRevenue por mês/trimestre | Window SUM acumulado |
| 8 | Produtos com reposição necessária | Último snapshot |
| 9 | Desconto médio por categoria e impacto | Métricas de desconto |
| 10 | Lead time médio por país de destino | Accumulating Snapshot |

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR

spark = get_spark("NorthwindDW SQL - 08 Analytics")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:54:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 11}


In [3]:
# ── Pergunta de negócio: "Qual categoria gera mais receita e qual o impacto dos descontos?"
# ── Métrica: GrossRevenue vs NetRevenue por CategoryName
# ── Fonte: FactSales ⋈ DimProduct
#
print("Query 1 — Receita por Categoria")
spark.sql("""
    SELECT p.CategoryName,
           COUNT(DISTINCT f.OrderID)                       AS Pedidos,
           ROUND(SUM(f.GrossRevenue), 2)                   AS GrossRevenue,
           ROUND(SUM(f.NetRevenue), 2)                     AS NetRevenue,
           ROUND(SUM(f.GrossRevenue - f.NetRevenue), 2)    AS DescontoTotal,
           ROUND(AVG(f.Discount) * 100, 2)                 AS DescontoMedioPct
    FROM gold.FactSales f
    JOIN gold.DimProduct p ON f.ProductSK = p.ProductSK
    GROUP BY p.CategoryName ORDER BY NetRevenue DESC
""").show(truncate=False)

Query 1 — Receita por Categoria


+--------------+-------+------------+----------+-------------+----------------+
|CategoryName  |Pedidos|GrossRevenue|NetRevenue|DescontoTotal|DescontoMedioPct|
+--------------+-------+------------+----------+-------------+----------------+
|Beverages     |354    |286526.95   |267868.18 |18658.77     |6.19            |
|Dairy Products|303    |251330.5    |234507.28 |16823.22     |5.34            |
|Confections   |295    |177099.1    |167357.22 |9741.88      |5.69            |
|Meat/Poultry  |161    |178188.8    |163022.36 |15166.44     |6.45            |
|Seafood       |291    |141623.09   |131261.74 |10361.35     |6.02            |
|Condiments    |193    |113694.75   |106047.08 |7647.67      |5.26            |
|Produce       |129    |105268.6    |99984.58  |5284.02      |4.54            |
|Grains/Cereals|182    |100726.8    |95744.59  |4982.21      |4.53            |
+--------------+-------+------------+----------+-------------+----------------+



In [4]:
# ── Pergunta de negócio: "Quem são nossos melhores clientes e quanto representam?"
# ── Métrica: NetRevenue total e % do total por CompanyName
# ── Fonte: FactSales ⋈ DimCustomer (perfil atual — IsCurrent=true)
#
print("Query 2 — Top 10 Clientes")
spark.sql("""
    SELECT c.CompanyName, c.Country,
           COUNT(DISTINCT f.OrderID)     AS Pedidos,
           ROUND(SUM(f.NetRevenue), 2)   AS NetRevenue,
           ROUND(AVG(f.NetRevenue), 2)   AS TicketMedio
    FROM gold.FactSales f
    JOIN gold.DimCustomer c ON f.CustomerSK = c.CustomerSK
    GROUP BY c.CompanyName, c.Country ORDER BY NetRevenue DESC LIMIT 10
""").show(truncate=False)

Query 2 — Top 10 Clientes


+----------------------------+-------+-------+----------+-----------+
|CompanyName                 |Country|Pedidos|NetRevenue|TicketMedio|
+----------------------------+-------+-------+----------+-----------+
|QUICK-Stop                  |Germany|28     |110277.3  |1282.29    |
|Ernst Handel                |Austria|30     |104874.98 |1028.19    |
|Save-a-lot Markets          |USA    |31     |104361.95 |899.67     |
|Rattlesnake Canyon Grocery  |USA    |18     |51097.8   |719.69     |
|Hungry Owl All-Night Grocers|Ireland|19     |49979.9   |908.73     |
|Hanari Carnes               |Brazil |14     |32841.37  |1026.29    |
|Königlich Essen             |Germany|14     |30908.38  |792.52     |
|Folk och fä HB              |Sweden |19     |29567.56  |657.06     |
|Mère Paillarde              |Canada |13     |28872.19  |902.26     |
|White Clover Markets        |USA    |14     |27363.6   |684.09     |
+----------------------------+-------+-------+----------+-----------+



In [5]:
# ── Pergunta de negócio: "Qual transportadora cumpre melhor o prazo de entrega?"
# ── Métrica: DaysToShip médio + % de pedidos em atraso (IsLate)
# ── Fonte: FactOrderFulfillment (Accumulating Snapshot) ⋈ DimShipper
#
print("Query 3 — Performance de Entrega por Shipper")
spark.sql("""
    SELECT s.CompanyName AS Shipper,
           COUNT(*) AS TotalEntregues,
           SUM(CASE WHEN f.IsLate = false THEN 1 ELSE 0 END) AS OnTime,
           SUM(CASE WHEN f.IsLate = true  THEN 1 ELSE 0 END) AS Atrasados,
           ROUND(100.0 * SUM(CASE WHEN f.IsLate = false THEN 1 ELSE 0 END) / COUNT(*), 1) AS OnTimePct,
           ROUND(AVG(CAST(f.DaysToShip AS DOUBLE)), 1) AS MediaDiasEnvio
    FROM gold.FactOrderFulfillment f
    JOIN gold.DimShipper s ON f.ShipperSK = s.ShipperSK
    WHERE f.ShippedDateKey IS NOT NULL
    GROUP BY s.CompanyName ORDER BY OnTimePct DESC
""").show(truncate=False)

Query 3 — Performance de Entrega por Shipper


+----------------+--------------+------+---------+---------+--------------+
|Shipper         |TotalEntregues|OnTime|Atrasados|OnTimePct|MediaDiasEnvio|
+----------------+--------------+------+---------+---------+--------------+
|Federal Shipping|249           |240   |9        |96.4     |7.5           |
|Speedy Express  |245           |233   |12       |95.1     |8.6           |
|United Package  |315           |299   |16       |94.9     |9.2           |
+----------------+--------------+------+---------+---------+--------------+



In [6]:
# ── Pergunta de negócio: "De onde vinha a receita historicamente? (não onde o cliente está hoje)"
# ── Métrica: NetRevenue por Country NA DATA DO PEDIDO — demonstra o valor do SCD2
# ── Fonte: FactSales ⋈ DimCustomer (todas as versões, sem filtro IsCurrent)
#
print("Query 4 — Receita por País Histórico do Cliente (SCD2 awareness)")
spark.sql("""
    SELECT c.Country AS PaisNaMomentoVenda, d.Year AS Ano,
           COUNT(DISTINCT f.OrderID) AS Pedidos,
           ROUND(SUM(f.NetRevenue), 2) AS NetRevenue,
           RANK() OVER (PARTITION BY d.Year ORDER BY SUM(f.NetRevenue) DESC) AS Rank
    FROM gold.FactSales f
    JOIN gold.DimCustomer c ON f.CustomerSK = c.CustomerSK
    JOIN gold.DimDate d ON f.OrderDateKey = d.DateKey
    GROUP BY c.Country, d.Year ORDER BY d.Year, Rank
""").show(30, truncate=False)

Query 4 — Receita por País Histórico do Cliente (SCD2 awareness)


+------------------+----+-------+----------+----+
|PaisNaMomentoVenda|Ano |Pedidos|NetRevenue|Rank|
+------------------+----+-------+----------+----+
|USA               |1996|23     |38105.67  |1   |
|Germany           |1996|24     |35407.14  |2   |
|Austria           |1996|8      |25601.34  |3   |
|Brazil            |1996|13     |20148.82  |4   |
|France            |1996|15     |17372.76  |5   |
|Venezuela         |1996|8      |9738.1    |6   |
|UK                |1996|10     |9273.68   |7   |
|Ireland           |1996|5      |9123.38   |8   |
|Canada            |1996|4      |7372.68   |9   |
|Sweden            |1996|6      |6933.22   |10  |
|Belgium           |1996|2      |6306.7    |11  |
|Mexico            |1996|9      |4687.9    |12  |
|Switzerland       |1996|3      |4164.72   |13  |
|Finland           |1996|4      |3115.76   |14  |
|Spain             |1996|6      |2976.2    |15  |
|Denmark           |1996|3      |2952.4    |16  |
|Portugal          |1996|4      |2306.14   |17  |


In [7]:
# ── Pergunta de negócio: "Como a variação de preço ao longo do tempo afetou a receita?"
# ── Métrica: UnitPrice por versão SCD2 + NetRevenue acumulado em cada vigência
# ── Fonte: FactSales ⋈ DimProduct (todas as versões — rastreia mudança de preço)
#
print("Query 5 — Histórico de Preço por Produto (versões SCD2)")
spark.sql("""
    SELECT p.ProductName, p.CategoryName, p.UnitPrice AS PrecoNaVersao,
           p.ValidFrom, p.ValidTo,
           COUNT(f.SalesSK) AS Vendas,
           ROUND(COALESCE(SUM(f.NetRevenue), 0), 2) AS ReceitaNaVersao
    FROM gold.DimProduct p
    LEFT JOIN gold.FactSales f ON f.ProductSK = p.ProductSK
    GROUP BY p.ProductName, p.CategoryName, p.UnitPrice, p.ValidFrom, p.ValidTo
    ORDER BY p.ProductName, p.ValidFrom
""").show(20, truncate=False)

Query 5 — Histórico de Preço por Produto (versões SCD2)


+----------------------------+--------------+-------------+----------+----------+------+---------------+
|ProductName                 |CategoryName  |PrecoNaVersao|ValidFrom |ValidTo   |Vendas|ReceitaNaVersao|
+----------------------------+--------------+-------------+----------+----------+------+---------------+
|Alice Mutton                |Meat/Poultry  |39.0         |1900-01-01|9999-12-31|37    |32698.38       |
|Aniseed Syrup               |Condiments    |10.0         |1900-01-01|9999-12-31|12    |3044.0         |
|Boston Crab Meat            |Seafood       |18.4         |1900-01-01|9999-12-31|41    |17910.63       |
|Camembert Pierrot           |Dairy Products|34.0         |1900-01-01|9999-12-31|51    |46825.48       |
|Carnarvon Tigers            |Seafood       |62.5         |1900-01-01|9999-12-31|27    |29171.87       |
|Chai                        |Beverages     |18.0         |1900-01-01|9999-12-31|38    |12788.1        |
|Chang                       |Beverages     |19.0      

In [8]:
# ── Pergunta de negócio: "Qual a contribuição de receita por gestor e sua equipe?"
# ── Métrica: NetRevenue por gestor (hierarquia achatada DimEmployee.ManagerName)
# ── Fonte: FactSales ⋈ DimEmployee (self-join achatado — sem CTE recursiva)
#
print("Query 6 — Hierarquia de Funcionários: Receita por Gestor")
spark.sql("""
    SELECT COALESCE(e.ManagerName, '(Presidente)') AS Gestor,
           e.FullName AS Funcionario, e.Title,
           COUNT(f.SalesSK) AS Transacoes,
           ROUND(SUM(f.NetRevenue), 2) AS NetRevenue,
           ROUND(AVG(f.NetRevenue), 2) AS TicketMedio
    FROM gold.FactSales f
    JOIN gold.DimEmployee e ON f.EmployeeSK = e.EmployeeSK
    GROUP BY e.ManagerName, e.FullName, e.Title
    ORDER BY Gestor, NetRevenue DESC
""").show(truncate=False)

Query 6 — Hierarquia de Funcionários: Receita por Gestor


+---------------+----------------+------------------------+----------+----------+-----------+
|Gestor         |Funcionario     |Title                   |Transacoes|NetRevenue|TicketMedio|
+---------------+----------------+------------------------+----------+----------+-----------+
|(Presidente)   |Andrew Fuller   |Vice President, Sales   |241       |166537.75 |691.03     |
|Andrew Fuller  |Margaret Peacock|Sales Representative    |420       |232890.85 |554.5      |
|Andrew Fuller  |Janet Leverling |Sales Representative    |321       |202812.84 |631.82     |
|Andrew Fuller  |Nancy Davolio   |Sales Representative    |345       |192107.6  |556.83     |
|Andrew Fuller  |Laura Callahan  |Inside Sales Coordinator|260       |126862.28 |487.93     |
|Andrew Fuller  |Steven Buchanan |Sales Manager           |117       |68792.28  |587.97     |
|Steven Buchanan|Robert King     |Sales Representative    |176       |124568.23 |707.77     |
|Steven Buchanan|Anne Dodsworth  |Sales Representative    |1

In [9]:
# ── Pergunta de negócio: "A receita é sazonal? Há meses/trimestres de pico?"
# ── Métrica: NetRevenue mensal + acumulado no trimestre (window function)
# ── Fonte: FactSales ⋈ DimDate
#
print("Query 7 — Sazonalidade: NetRevenue por Mês (com acumulado no trimestre)")
spark.sql("""
    SELECT d.Year, d.Quarter, d.Month, d.MonthName,
           COUNT(DISTINCT f.OrderID) AS Pedidos,
           ROUND(SUM(f.NetRevenue), 2) AS NetRevenue,
           ROUND(SUM(SUM(f.NetRevenue)) OVER (
               PARTITION BY d.Year, d.Quarter ORDER BY d.Month
           ), 2) AS AcumuladoTrimestre
    FROM gold.FactSales f
    JOIN gold.DimDate d ON f.OrderDateKey = d.DateKey
    GROUP BY d.Year, d.Quarter, d.Month, d.MonthName
    ORDER BY d.Year, d.Month
""").show(40, truncate=False)

Query 7 — Sazonalidade: NetRevenue por Mês (com acumulado no trimestre)


+----+-------+-----+---------+-------+----------+------------------+
|Year|Quarter|Month|MonthName|Pedidos|NetRevenue|AcumuladoTrimestre|
+----+-------+-----+---------+-------+----------+------------------+
|1996|3      |7    |July     |22     |27861.89  |27861.89          |
|1996|3      |8    |August   |25     |25485.27  |53347.17          |
|1996|3      |9    |September|23     |26381.4   |79728.57          |
|1996|4      |10   |October  |26     |37515.72  |37515.72          |
|1996|4      |11   |November |25     |45600.04  |83115.77          |
|1996|4      |12   |December |31     |45239.63  |128355.4          |
|1997|1      |1    |January  |33     |61258.07  |61258.07          |
|1997|1      |2    |February |29     |38483.63  |99741.7           |
|1997|1      |3    |March    |30     |38547.22  |138288.92         |
|1997|2      |4    |April    |31     |53032.95  |53032.95          |
|1997|2      |5    |May      |32     |53781.29  |106814.24         |
|1997|2      |6    |June     |30  

In [10]:
# ── Pergunta de negócio: "Quais produtos estão abaixo do nível de reposição agora?"
# ── Métrica: UnitsInStock vs ReorderLevel no último snapshot de estoque
# ── Fonte: FactProductStock (Periodic Snapshot — último RunID) ⋈ DimProduct
#
print("Query 8 — Produtos que Precisam de Reposição")
spark.sql("""
    SELECT p.ProductName, p.CategoryName,
           fs.UnitsInStock, fs.ReorderLevel, fs.UnitsOnOrder,
           fs.UnitsInStock - fs.ReorderLevel AS EstoqueAcimaReorder,
           fs.SnapshotDateKey
    FROM gold.FactProductStock fs
    JOIN gold.DimProduct p ON fs.ProductSK = p.ProductSK
    WHERE fs.NeedsReorder = true
      AND fs.SnapshotDateKey = (SELECT MAX(SnapshotDateKey) FROM gold.FactProductStock)
    ORDER BY EstoqueAcimaReorder
""").show(30, truncate=False)

Query 8 — Produtos que Precisam de Reposição


+-------------------------+--------------+------------+------------+------------+-------------------+---------------+
|ProductName              |CategoryName  |UnitsInStock|ReorderLevel|UnitsOnOrder|EstoqueAcimaReorder|SnapshotDateKey|
+-------------------------+--------------+------------+------------+------------+-------------------+---------------+
|Gorgonzola Telino        |Dairy Products|0           |20          |70          |-20                |20260329       |
|Mascarpone Fabioli       |Dairy Products|9           |25          |40          |-16                |20260329       |
|Louisiana Hot Spiced Okra|Condiments    |4           |20          |100         |-16                |20260329       |
|Outback Lager            |Beverages     |15          |30          |10          |-15                |20260329       |
|Gravad lax               |Seafood       |11          |25          |50          |-14                |20260329       |
|Aniseed Syrup            |Condiments    |13          |2

In [11]:
# ── Pergunta de negócio: "Em quais categorias os descontos mais corroem a margem?"
# ── Métrica: Desconto médio (%) + GrossRevenue vs NetRevenue por CategoryName
# ── Fonte: FactSales ⋈ DimProduct
#
print("Query 9 — Impacto do Desconto por Categoria")
spark.sql("""
    SELECT p.CategoryName,
           ROUND(AVG(f.Discount) * 100, 2) AS DescontoMedioPct,
           ROUND(SUM(f.GrossRevenue), 2) AS GrossRevenue,
           ROUND(SUM(f.NetRevenue), 2) AS NetRevenue,
           ROUND(SUM(f.GrossRevenue - f.NetRevenue), 2) AS ReceitaPerdida,
           ROUND(100.0 * SUM(f.GrossRevenue - f.NetRevenue) / SUM(f.GrossRevenue), 2) AS PctPerdida
    FROM gold.FactSales f
    JOIN gold.DimProduct p ON f.ProductSK = p.ProductSK
    GROUP BY p.CategoryName ORDER BY PctPerdida DESC
""").show(truncate=False)

Query 9 — Impacto do Desconto por Categoria


+--------------+----------------+------------+----------+--------------+----------+
|CategoryName  |DescontoMedioPct|GrossRevenue|NetRevenue|ReceitaPerdida|PctPerdida|
+--------------+----------------+------------+----------+--------------+----------+
|Meat/Poultry  |6.45            |178188.8    |163022.36 |15166.44      |8.51      |
|Seafood       |6.02            |141623.09   |131261.74 |10361.35      |7.32      |
|Condiments    |5.26            |113694.75   |106047.08 |7647.67       |6.73      |
|Dairy Products|5.34            |251330.5    |234507.28 |16823.22      |6.69      |
|Beverages     |6.19            |286526.95   |267868.18 |18658.77      |6.51      |
|Confections   |5.69            |177099.1    |167357.22 |9741.88       |5.5       |
|Produce       |4.54            |105268.6    |99984.58  |5284.02       |5.02      |
|Grains/Cereals|4.53            |100726.8    |95744.59  |4982.21       |4.95      |
+--------------+----------------+------------+----------+--------------+----

In [12]:
# ── Pergunta de negócio: "Para quais destinos os pedidos demoram mais para chegar?"
# ── Métrica: DaysToShip médio + % pontualidade por ShipCountry
# ── Fonte: FactOrderFulfillment (Accumulating Snapshot)
#
print("Query 10 — Pedidos por País de Destino (Lead Time + Pontualidade)")
spark.sql("""
    SELECT f.ShipCountry,
           COUNT(DISTINCT f.OrderID) AS TotalPedidos,
           SUM(CASE WHEN f.IsLate = false THEN 1 ELSE 0 END) AS Pontuais,
           SUM(CASE WHEN f.IsLate = true  THEN 1 ELSE 0 END) AS Atrasados,
           ROUND(AVG(CAST(f.DaysToShip AS DOUBLE)), 1) AS LeadTimeMedio,
           ROUND(100.0 * SUM(CASE WHEN f.IsLate = false THEN 1 ELSE 0 END)
                 / NULLIF(COUNT(CASE WHEN f.ShippedDateKey IS NOT NULL THEN 1 END), 0), 1) AS OnTimePct
    FROM gold.FactOrderFulfillment f
    GROUP BY f.ShipCountry ORDER BY TotalPedidos DESC
""").show(30, truncate=False)

Query 10 — Pedidos por País de Destino (Lead Time + Pontualidade)


+-----------+------------+--------+---------+-------------+---------+
|ShipCountry|TotalPedidos|Pontuais|Atrasados|LeadTimeMedio|OnTimePct|
+-----------+------------+--------+---------+-------------+---------+
|Germany    |122         |116     |4        |8.0          |96.7     |
|USA        |122         |112     |7        |9.6          |94.1     |
|Brazil     |83          |78      |3        |8.1          |96.3     |
|France     |77          |72      |3        |8.4          |96.0     |
|UK         |56          |52      |4        |8.2          |92.9     |
|Venezuela  |46          |41      |2        |8.5          |95.3     |
|Austria    |40          |37      |1        |8.7          |97.4     |
|Sweden     |37          |34      |3        |10.2         |91.9     |
|Canada     |30          |29      |0        |5.9          |100.0    |
|Mexico     |28          |27      |0        |7.8          |100.0    |
|Italy      |28          |25      |2        |7.9          |92.6     |
|Spain      |23     

In [13]:
print("BÔNUS — Delta Time Travel")
spark.sql("DESCRIBE HISTORY gold.FactSales").select("version", "timestamp", "operation").show(5, truncate=False)
print("\nPara versão histórica: spark.read.format('delta').option('versionAsOf', 0).table('gold.FactSales')")

BÔNUS — Delta Time Travel


+-------+-----------------------+------------+
|version|timestamp              |operation   |
+-------+-----------------------+------------+
|1      |2026-03-29 00:50:55.849|MERGE       |
|0      |2026-03-29 00:42:41.288|CREATE TABLE|
+-------+-----------------------+------------+


Para versão histórica: spark.read.format('delta').option('versionAsOf', 0).table('gold.FactSales')
